In [ ]:
"""
PDF Web Scraper
===============
Download PDF files from a webpage containing list of PDF links.

Features:
- Scrape webpage for PDF links
- Download PDFs to data/pdf_books folder
- Skip already downloaded files
- Automatically process and add to database

Usage:
    python -m src.scrapper_transcriber.pdf_scraper <url>
    python -m src.scrapper_transcriber.pdf_scraper <url> --process

Example:
    python -m src.scrapper_transcriber.pdf_scraper "https://example.com/books"

Requirements:
    pip install requests beautifulsoup4
"""

In [ ]:
import os
import re
import time
import hashlib
import argparse
from typing import List, Dict, Optional
from urllib.parse import urljoin, urlparse, unquote
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [ ]:
# Configuration
# __file__ is not available in Jupyter notebooks, use os.getcwd() instead
SCRIPT_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PDF_FOLDER = os.path.join(DATA_DIR, "pdf_books")

In [ ]:
os.makedirs(PDF_FOLDER, exist_ok=True)

In [ ]:
# Request settings
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

In [ ]:
REQUEST_TIMEOUT = 30
DOWNLOAD_CHUNK_SIZE = 8192
RETRY_ATTEMPTS = 3
RETRY_DELAY = 2

In [ ]:
def get_filename_from_url(url: str) -> str:
    """Extract filename from URL."""
    parsed = urlparse(url)
    path = unquote(parsed.path)
    filename = os.path.basename(path)
    
    # Ensure it ends with .pdf
    if not filename.lower().endswith('.pdf'):
        filename += '.pdf'
    
    # Clean filename
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    
    return filename

In [ ]:
def get_filename_from_headers(response) -> Optional[str]:
    """Extract filename from Content-Disposition header."""
    cd = response.headers.get('Content-Disposition')
    if cd:
        # Try to extract filename
        match = re.search(r'filename[^;=\n]*=(([\'"]).*?\2|[^\s;]+)', cd)
        if match:
            filename = match.group(1).strip('"\'')
            return filename
    return None

In [ ]:
def scrape_pdf_links(url: str, same_domain_only: bool = False) -> List[Dict]:
    """
    Scrape webpage for PDF links.
    
    Args:
        url: Webpage URL to scrape
        same_domain_only: Only return links from same domain
        
    Returns:
        List of dicts with 'url' and 'text' keys
    """
    print(f"🔍 Scraping: {url}")
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
    except Exception as e:
        print(f"❌ Error fetching page: {e}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    base_domain = urlparse(url).netloc
    
    pdf_links = []
    seen_urls = set()
    
    # Find all links
    for link in soup.find_all('a', href=True):
        href = link['href']
        
        # Make absolute URL
        full_url = urljoin(url, href)
        
        # Check if it's a PDF
        is_pdf = (
            href.lower().endswith('.pdf') or
            'pdf' in href.lower() or
            link.get_text().lower().endswith('.pdf')
        )
        
        if not is_pdf:
            continue
        
        # Check domain if required
        if same_domain_only:
            link_domain = urlparse(full_url).netloc
            if link_domain != base_domain:
                continue
        
        # Skip duplicates
        if full_url in seen_urls:
            continue
        seen_urls.add(full_url)
        
        # Get link text
        text = link.get_text().strip()
        if not text:
            text = get_filename_from_url(full_url)
        
        pdf_links.append({
            'url': full_url,
            'text': text
        })
    
    print(f"   Found {len(pdf_links)} PDF links")
    return pdf_links

In [ ]:
def download_pdf(url: str, output_dir: str = PDF_FOLDER, skip_existing: bool = True) -> Optional[str]:
    """
    Download a PDF file.
    
    Args:
        url: PDF URL
        output_dir: Directory to save to
        skip_existing: Skip if file already exists
        
    Returns:
        Path to downloaded file, or None if failed
    """
    # Get filename
    filename = get_filename_from_url(url)
    output_path = os.path.join(output_dir, filename)
    
    # Check if exists
    if skip_existing and os.path.exists(output_path):
        print(f"   ⏭️ Already exists: {filename}")
        return output_path
    
    print(f"   📥 Downloading: {filename}")
    
    for attempt in range(RETRY_ATTEMPTS):
        try:
            response = requests.get(
                url, 
                headers=HEADERS, 
                timeout=REQUEST_TIMEOUT,
                stream=True
            )
            response.raise_for_status()
            
            # Check content type
            content_type = response.headers.get('Content-Type', '')
            if 'pdf' not in content_type.lower() and 'octet-stream' not in content_type.lower():
                print(f"   ⚠️ Not a PDF: {content_type}")
                return None
            
            # Try to get filename from headers
            header_filename = get_filename_from_headers(response)
            if header_filename:
                filename = re.sub(r'[<>:"/\\|?*]', '_', header_filename)
                output_path = os.path.join(output_dir, filename)
            
            # Get total size
            total_size = int(response.headers.get('Content-Length', 0))
            
            # Download with progress
            with open(output_path, 'wb') as f:
                if total_size > 0:
                    with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"      {filename[:30]}") as pbar:
                        for chunk in response.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
                            if chunk:
                                f.write(chunk)
                                pbar.update(len(chunk))
                else:
                    for chunk in response.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
            
            print(f"   ✅ Downloaded: {filename}")
            return output_path
            
        except Exception as e:
            print(f"   ⚠️ Attempt {attempt + 1}/{RETRY_ATTEMPTS} failed: {e}")
            if attempt < RETRY_ATTEMPTS - 1:
                time.sleep(RETRY_DELAY)
    
    print(f"   ❌ Failed to download: {url}")
    return None

In [ ]:
def scrape_and_download(
    url: str, 
    output_dir: str = PDF_FOLDER,
    same_domain_only: bool = False,
    skip_existing: bool = True,
    process_after: bool = False
) -> List[str]:
    """
    Scrape webpage for PDFs and download them.
    
    Args:
        url: Webpage URL
        output_dir: Directory to save PDFs
        same_domain_only: Only download from same domain
        skip_existing: Skip already downloaded files
        process_after: Process PDFs after download
        
    Returns:
        List of downloaded file paths
    """
    print("=" * 60)
    print("PDF Web Scraper")
    print("=" * 60)
    
    # Scrape links
    pdf_links = scrape_pdf_links(url, same_domain_only)
    
    if not pdf_links:
        print("❌ No PDF links found")
        return []
    
    # Download each
    downloaded = []
    print(f"\n📥 Downloading {len(pdf_links)} PDFs...")
    
    for i, link in enumerate(pdf_links, 1):
        print(f"\n[{i}/{len(pdf_links)}] {link['text'][:50]}")
        path = download_pdf(link['url'], output_dir, skip_existing)
        if path:
            downloaded.append(path)
        
        # Small delay between downloads
        time.sleep(0.5)
    
    print("\n" + "=" * 60)
    print(f"✅ Downloaded {len(downloaded)}/{len(pdf_links)} PDFs")
    print(f"   Location: {output_dir}")
    print("=" * 60)
    
    # Process if requested
    if process_after and downloaded:
        print("\n📖 Processing downloaded PDFs...")
        from src.scrapper_transcriber.pdf_transcribe import process_pdf
        
        for pdf_path in downloaded:
            process_pdf(pdf_path)
    
    return downloaded

In [ ]:
def download_single_pdf(url: str, output_dir: str = PDF_FOLDER, process_after: bool = False) -> Optional[str]:
    """Download a single PDF by URL."""
    print("=" * 60)
    print("PDF Downloader")
    print("=" * 60)
    
    path = download_pdf(url, output_dir, skip_existing=True)
    
    if path and process_after:
        print("\n📖 Processing PDF...")
        from src.scrapper_transcriber.pdf_transcribe import process_pdf
        process_pdf(path)
    
    return path

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="PDF Web Scraper")
    parser.add_argument("url", help="URL of webpage with PDF links, or direct PDF URL")
    parser.add_argument("--output", "-o", default=PDF_FOLDER, help="Output directory")
    parser.add_argument("--same-domain", action="store_true", help="Only download from same domain")
    parser.add_argument("--force", "-f", action="store_true", help="Re-download existing files")
    parser.add_argument("--process", "-p", action="store_true", help="Process PDFs after download")
    parser.add_argument("--direct", "-d", action="store_true", help="URL is direct PDF link")
    
    args = parser.parse_args()
    
    os.makedirs(args.output, exist_ok=True)
    
    if args.direct or args.url.lower().endswith('.pdf'):
        # Direct PDF download
        download_single_pdf(args.url, args.output, args.process)
    else:
        # Scrape page for PDF links
        scrape_and_download(
            args.url,
            args.output,
            same_domain_only=args.same_domain,
            skip_existing=not args.force,
            process_after=args.process
        )

In [ ]:
if __name__ == "__main__":
    main()